# IMPORT MODULES

In [7]:
import pandas as pd
import numpy as np
import random
import itertools
import json
import pprint
import datetime
from datetime import date, time, timedelta, datetime
import requests
import os
import pandas_market_calendars as mcal
import base64
from datetime import datetime, timedelta, date
import traceback
import glob
import sys
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '..')))
from functions import *
from config import *


In [2]:
# Download data into memory
if 'Ticker' in ticker_list.columns:
    # Create a new DataFrame with only the 'Tickers' column
    tickers_df = ticker_list[['Ticker']].dropna()
    # Convert to a list of unique tickers
    tickers = tickers_df['Ticker'].unique().tolist()  # Extract unique tickers
    print(tickers)  # Display the list of tickers
    print('SUCESS IF YOU SEE TICKERS!!! ^^^^^^^')
else:
    print("Column 'Ticker' not found in the DataFrame.")

['BL', 'ALKT', 'STEW', 'CRL', 'ZBRA', 'PASG', 'AX', 'KFY', 'JKS', 'HCC', 'MOD', 'LMND', 'AIZ', 'TRMD', 'SNA', 'THS', 'AREC', 'CENX', 'VSTO', 'GDOT', 'WFG', 'DRTS', 'WDI', 'TY', 'SGML', 'CBU', 'CLYM', 'EBS', 'MHK', 'HG', 'VBTX', 'PLTK', 'MGTX', 'TFSL', 'GRNT', 'OPI', 'RGEN', 'VTEX', 'ACP', 'MHD', 'ZG', 'PII', 'ZVRA', 'EOS', 'REPL', 'KLIC', 'ARW', 'HYPR', 'MLTX', 'SOL', 'CENTA', 'QNST', 'FULC', 'PCN', 'SANM', 'PRQR', 'SUPN', 'IQI', 'RGLS', 'GKOS', 'DXLG', 'IDA', 'NNOX', 'SLRC', 'BTSG', 'MIDD', 'HLI', 'MATV', 'PRIM', 'PAVS', 'EXFY', 'CRUS', 'MUJ', 'CION', 'VUZI', 'CNTB', 'BAP', 'FIVE', 'WHR', 'WIX', 'EPAC', 'HUBB', 'BHAT', 'CRTO', 'ARBK', 'PLRX', 'CHW', 'PRTS', 'PHR', 'CBT', 'AHT', 'BHK', 'EMBC', 'ELF', 'LIVN', 'NDSN', 'LECO', 'REYN', 'VGM', 'GRWG', 'PAM', 'BUSE', 'FUL', 'ACB', 'EOLS', 'ENV', 'RYTM', 'CTOS', 'KTB', 'BZUN', 'MMU', 'IVR', 'RS', 'IBOC', 'VTYX', 'TDG', 'PXLW', 'GPCR', 'PARR', 'NMCO', 'RNST', 'NMRA', 'ONL', 'LUNR', 'PTLO', 'EPAM', 'ABVX', 'ENSG', 'AGIO', 'ENOV', 'VNDA', 'RA', 

In [3]:
# tickers = ['IONQ']  # Can easily change this to test different tickers
# print(f"Testing strategy on ticker: {tickers[0]}")

In [4]:
################################## TESTING/AUTHENTICATING CONNECTION TO SCHWAB API ##################################
# Get access token (figure out how often to do this optimally)
access_token = auto_authenticate(appKey, appSecret)
ACCOUNT_SIZE = get_account_balance()
# Get AAPL price history for the last year using the defined variables
aapl_price_history = get_stock_price_history('AAPL', access_token, period_type, period, frequency_type, frequency,  start_time, end_time,need_extended_hours_data, need_previous_close)

# Convert to DataFrame
aapl_df = pd.DataFrame(aapl_price_history['candles'])

# Rename columns to match the required format
aapl_df.rename(columns={
    'datetime': 'Datetime',
    'open': 'Open',
    'high': 'High',
    'low': 'Low',
    'close': 'Close',
    'volume': 'Volume'
}, inplace=True)

# Convert Datetime to EST
aapl_df['Datetime'] = pd.to_datetime(aapl_df['Datetime'], unit='ms').dt.tz_localize('UTC').dt.tz_convert('US/Eastern').dt.tz_localize(None)

# Save the DataFrame as a CSV file
# aapl_df.to_csv('./file_uploads/tests/aapl_price_history.csv', index=False)

# Print the result

# print(aapl_price_history)
print('SUCCESS IF I SEE APPLE STOCK DATA!!!')
# print('AAPL price history saved to ./file_uploads/tests/aapl_price_history.csv')

SUCCESS IF I SEE APPLE STOCK DATA!!!


# Get All Stock Data, Calculate Relevant Stats, and Store in DF's

In [5]:

######################### CREATE INITIAL DATAFRAME OF STOCK PRICE HISTORY AND PERFORM PRECALCULATIONS #########################

start_clock = datetime.now()  # calculate run time
go = 1
stockies = {} #Create dataframes of stock data for iteration

for ticker in tickers:
    # Get stock price history from Schwab API
    stock_data = get_stock_price_history(ticker, access_token, period_type, period, frequency_type, frequency, start_time, end_time, need_extended_hours_data, need_previous_close)
    
    if not stock_data or 'candles' not in stock_data:
        print(f"No data available for {ticker}")
        continue
    
    # Convert to DataFrame
    stahks = pd.DataFrame(stock_data['candles'])
    
    # Ensure all required fields are present
    required_fields = ['datetime', 'open', 'high', 'low', 'close', 'volume']
    if not all(field in stahks.columns for field in required_fields):
        print(f"Missing required fields for {ticker}. Available columns: {stahks.columns}")
        continue
    
    # Rename columns to match the required format
    stahks.rename(columns={
        'datetime': 'Datetime',
        'open': 'Open',
        'high': 'High',
        'low': 'Low',
        'close': 'Close',
        'volume': 'Volume'
    }, inplace=True)
    
    # Convert Datetime to EST
    stahks['Datetime'] = pd.to_datetime(stahks['Datetime'], unit='ms').dt.tz_localize('UTC').dt.tz_convert('US/Eastern').dt.tz_localize(None)
    
    # Print the DataFrame to inspect its structure
    # print(f"Data for {ticker}:")
    # print(stahks.head())  # Display the first few rows of the DataFrame
    # print("Columns in DataFrame:", stahks.columns)  # Print the column names
    
    #Skip stock if there is insufficient amount of data (data for each period of normal trading hours)
    end_check = stahks['Datetime'].max()
    start_check = stahks['Datetime'].min()
    daydiff = end_check.weekday() - start_check.weekday()
    days = ((end_check-start_check).days - daydiff) / 7 * 5 + min(daydiff,5) - (max(end_check.weekday() - 4, 0) % 5)
    
    print(ticker, len(stahks.index))    
    # Removed the erroneous line that referenced 'datetime' instead of 'Datetime'
    # stahks['datetime'] = pd.to_datetime(stahks['datetime']/1000, unit = 's')-timedelta(hours =4)
    # stahks.columns = ['Open','High','Low','Close','Volume','Datetime']
    
    #Rearrange Columns and Merge with Open/Close Schedule
    stahks['Ticker'] = ticker
    stahks['Date'] = stahks['Datetime'].dt.date
    stahks = stahks.merge(open_close_schedule,how = 'left', on = 'Date')

    # Only calculate and average volume if there is sufficient data
    rolling_lookback_int = int(rolling_lookback)
    stahks['10_Day_Avg_Vol'] = stahks.Volume.rolling(rolling_lookback_int, min_periods=rolling_lookback_int).mean()
    stahks['10_Day_Avg_Vol'] = stahks['10_Day_Avg_Vol'].fillna(float('inf'))
    stahks['Time'] = stahks['Datetime'].dt.time
    
    #Remove any accidental duplicates (FIGURE OUT WHY????)
    stahks.drop_duplicates(['Ticker','Date','Time'],inplace = True,ignore_index=True)
    
    if len(stahks.index) < (days * tickers_per_day):
        continue
        
    # Create column with the open bar's low price (For % gap up calculation with spike later in day)
    cond = (stahks['Time'] == stahks['market_open'])
    stahks['Day_Open_Low'] = stahks[cond].groupby('Date', as_index=True)['Low'].transform('min').ffill()

    stahks['After Hours'] = (stahks['Time'] > stahks['market_close']) | (stahks['Time'] < stahks['market_open'])

    cond_2 = (stahks['After Hours'] == True)
    stahks['Pre-Market High'] = stahks[cond_2].groupby('Date', as_index=True)['High'].transform('max')
    
    stahks = stahks.ffill(axis=0)
    stahks = stahks.bfill(axis=0)

    
    stahks['VWAP_Row'] = stahks['Volume']*((stahks['High']+stahks['Low']+stahks['Close'])/3)
    stahks['Cum_VWAP'] = stahks.groupby('Date')['VWAP_Row'].transform('cumsum')
    stahks['Cum_Volume'] = stahks.groupby('Date')['Volume'].transform('cumsum')
    stahks['VWAP'] = stahks['Cum_VWAP']/stahks['Cum_Volume']
    stahks['VWAP_STD_1'] = stahks['VWAP'] - stahks.groupby('Date')['VWAP'].transform('std')
    stahks['Color_Bar'] = np.where(stahks['Open']<=stahks['Close'], 'Green', 'Red')
    #vol_window = 1
    stahks['Day_Close'] = (stahks['Time'] == stahks['market_close'])

    
    stockies[ticker] = pd.DataFrame(stahks, columns=stahks.keys())
    print(f"{ticker} processed successfully.")
    go += 1

    #Calculate pre-market volume for day 
    #Calculate pre-market change for the day 
    #stockies['Ticker_Return'] = (stahks['Close']/stahks['Close'].shift(vol_window))-1
    #stockies['Rolling_Vol'] = stahks['Ticker_Return'].std(ddof=130)
    
    ##PRINT OUT THE DATAFRAME TO A CSV FILE
    # stahks.to_csv(f"screener_pre.csv", index=False, header=True)

#Stockies is the final dataframe that will be used for the backtest




BL 218
BL processed successfully.
ALKT 247
ALKT processed successfully.
STEW 212
STEW processed successfully.
CRL 239
CRL processed successfully.
ZBRA 222
ZBRA processed successfully.
PASG 246
PASG processed successfully.
AX 223
AX processed successfully.
KFY 217
KFY processed successfully.
JKS 424
JKS processed successfully.
HCC 241
HCC processed successfully.
MOD 236
MOD processed successfully.
LMND 334
LMND processed successfully.
AIZ 228
AIZ processed successfully.
TRMD 374
TRMD processed successfully.
SNA 226
SNA processed successfully.
THS 214
THS processed successfully.
AREC 248
AREC processed successfully.
CENX 243
CENX processed successfully.
VSTO 223
VSTO processed successfully.
GDOT 223
GDOT processed successfully.
WFG 213
WFG processed successfully.
DRTS 111
WDI 222
WDI processed successfully.
TY 192
TY processed successfully.
SGML 256
SGML processed successfully.
CBU 214
CBU processed successfully.
CLYM 217
CLYM processed successfully.
EBS 316
EBS processed successfully.
M

# DATA PROCESSING AND SIGNALING

In [11]:
RESULT_INDEXER = 0
COMBO_INDEXER = 0
stocks_to_trade = pd.DataFrame(columns=['Strategy','Date','Ticker','Target Entry','Volume Spike','Price Spike','Previous Day Close','Signal Time', 'Shares', 'Stop Price', 'Sell Price', 'Backup Sell Time'])

for strategy in combinations:
    print(strategy,(datetime.now() - start_clock))
    tickers = list(stockies.keys())
    for ticker in tickers:

###########################################  CALCULATE SIGNALS FOR BACKTEST  ###########################################

        #Checks for Highest Daily Volume
        high_vol_sig = np.where(stockies[ticker]['Volume'] == stockies[ticker].groupby('Date')['Volume'].transform('max'),'True','False')
        stockies[ticker]['high_vol_sig'] = high_vol_sig
        #Checks for price spike 10x greater than 
        price_sig = np.where((stockies[ticker]['High']-stockies[ticker]['Day_Open_Low'])/stockies[ticker]['Day_Open_Low'] >= strategy[price_spike_thresh_index],'True','False')
        stockies[ticker]['price_sig'] = price_sig
        #Checks that the spike was before 12 pm (tied to highest daily volume)
        before_time = np.where(stockies[ticker]['Time'] <= strategy[time_sig_thresh_index], 'True','False')
        stockies[ticker]['before_time'] = before_time 
        #Checks that spike was a green bar
        green_bar = np.where(stockies[ticker]['Color_Bar'] == 'Green', 'True','False')
        stockies[ticker]['green_bar'] = green_bar
        #Checks for volume spike 10x greater than 10day average
        vol_spike_sig = np.where(stockies[ticker]['Volume'] > strategy[vol_spike_thresh_index] * stockies[ticker]['10_Day_Avg_Vol'],'True','False')
        stockies[ticker]['vol_spike_sig'] = vol_spike_sig
        #Checks that the spike high was the highest price of the day
        high_price_sig = np.where(stockies[ticker]['High'] >= stockies[ticker].groupby('Date')['Close'].transform('max'),'True','False')
        stockies[ticker]['high_price_sig'] = high_price_sig
        #Checks to see if it time is equal to the sell_time threshold set by user
        stockies[ticker]['sell_time'] = np.where(stockies[ticker]['Time'] == strategy[sell_time_threshold_index],'True','False')
        #Finds High of the Day
        stockies[ticker]['high_of_day'] = stockies[ticker].groupby('Date')['High'].transform('max')

        #Checks all conditions
        stockies[ticker]['Grab_Price_Signal'] = np.where((high_vol_sig == 'True') 
                                                         & (vol_spike_sig == 'True') 
                                                         & (price_sig == 'True') 
                                                         & (high_price_sig == 'True') 
                                                         # & (green_bar == 'True') 
                                                         & (before_time == 'True'),
                                                         'True','False')

    ################################ INPUT BUY AND SELL SIGNALS  ##################################################

        #Find a way to turn on and off signals and rules to be 'True' 'False' 'Ignore' - yet still works with backtest framework

        #Checks that the close was lower than the VWAP (maybe in backtest)
        stockies[ticker]['Close_Condition'] = 'False'

        #Temp Variables for backtest (row by row iteration)
        #Create temporary signal day variable that signaled whether or not the price_to_buy_signal was triggered during that day
        TEMP_SIGNAL_DAY = stockies[ticker]['Date'][0] - timedelta(days=10)

        #Initially set not to trigger and gets set on price_buy_signal
        TARGET_ENTRY_PRICE = 0 
        TARGET_ENTRY_PRICE_2 = 0
        TEMP_SIGNAL_TIME = time(hour = 19, minute = 30, second = 0)
        
        VOLUME_SPIKE = 0
        PRICE_SPIKE = 0

        #TO DEBUG - WILL PRINT RAW DATA OF EACH STOCK WITH ALL PRECALCULATIONS AND SIGNALS
        # # Create the directory path
        # directory = f'./daily_signals/details/{str(date.today() + timedelta(days=1))}'
        # os.makedirs(directory, exist_ok=True)  # This will create all necessary parent directories
        
        # # Now you can safely save your files
        # output_file_path_ticker = f'{directory}/{ticker}_signals_.xlsx'
        # stockies[ticker].to_excel(output_file_path_ticker, index=False, header=True)

        #Iterate over rows to see which rows meet the close condition and the all clear to buy signal (pending final signal: price cross)
        #Unique to this strategy's backtest. Could be a part of inserting variables and signals before BACKTEST SECTION
        
        for index, row in stockies[ticker].iterrows(): 
            
            if (row['Grab_Price_Signal'] == 'True') & (row['Date'] == day_of_backtest.date()):
                TEMP_SIGNAL_DAY = row['Date']
                TARGET_ENTRY_PRICE = row['VWAP']
                TEMP_SIGNAL_TIME = row['Time']
                VOLUME_SPIKE = row['Volume']/row['10_Day_Avg_Vol']
                PRICE_SPIKE = (row['High']-row['Day_Open_Low'])/row['Day_Open_Low']
                YESTERDAY_HIGH = row['high_of_day']

            if (row['Close']<=TARGET_ENTRY_PRICE) & (row['Day_Close'] == True) & (row['Date'] == TEMP_SIGNAL_DAY):
                print('Got in thur')
                stocks_to_trade.at[RESULT_INDEXER,'Previous Day Close'] = row['Close']
                stocks_to_trade.at[RESULT_INDEXER,'Strategy'] = COMBO_INDEXER
                stocks_to_trade.at[RESULT_INDEXER,'Ticker'] = row['Ticker']
                stocks_to_trade.at[RESULT_INDEXER,'Shares'] = ACCOUNT_SIZE*ALLOCATION/TARGET_ENTRY_PRICE
                stocks_to_trade.at[RESULT_INDEXER,'Target Entry'] = TARGET_ENTRY_PRICE
                stocks_to_trade.at[RESULT_INDEXER,'Date'] = row['Date']
                stocks_to_trade.at[RESULT_INDEXER,'Signal Time'] = TEMP_SIGNAL_TIME
                stocks_to_trade.at[RESULT_INDEXER,'Volume Spike'] = VOLUME_SPIKE
                stocks_to_trade.at[RESULT_INDEXER,'Price Spike'] = PRICE_SPIKE
                stocks_to_trade.at[RESULT_INDEXER,'Time Threshold'] = strategy[time_sig_thresh_index]
                stocks_to_trade.at[RESULT_INDEXER,'Sell_Time'] = strategy[sell_time_threshold_index]
                
                # Calculate Stop Price and Sell Price
                stop_price = TARGET_ENTRY_PRICE + (TARGET_ENTRY_PRICE * strategy[stop_index])
                sell_price = TARGET_ENTRY_PRICE - (TARGET_ENTRY_PRICE * strategy[target_index])
                stocks_to_trade.at[RESULT_INDEXER,'Stop Price'] = stop_price
                stocks_to_trade.at[RESULT_INDEXER,'Sell Price'] = sell_price

                RESULT_INDEXER +=1                
    COMBO_INDEXER +=1 
    
stocks_to_trade = pd.merge(stocks_to_trade,ticker_list[['Ticker','Market Cap','Sector','Float']],on = 'Ticker', how = 'left')  
stocks_to_trade['Market Capitalization'] = (stocks_to_trade['Market Cap'].astype(float)/1000000).astype(str) + 'M'
stocks_to_trade['Shares Float'] = (stocks_to_trade['Float'].astype(float)/1000000).astype(str) + 'M'
stocks_to_trade.sort_values(by = 'Market Capitalization',ascending = True,inplace = True)
output_file_path = './daily_screener_signals/' + str(day_of_backtest.date() + timedelta(days=1))+'.xlsx'
stocks_to_trade.to_excel(output_file_path, index=True, header=True)

print(datetime.now() - start_clock)

(0.1, 0.1, 0.05, 5, 0.05, datetime.time(12, 30), datetime.time(10, 30), datetime.time(15, 30)) 0:17:07.226397
Got in thur
Got in thur
Got in thur
Got in thur
Got in thur
Got in thur
Got in thur
Got in thur
Got in thur
Got in thur
Got in thur
Got in thur
Got in thur
Got in thur
Got in thur
Got in thur
Got in thur
Got in thur
0:17:23.644835
